# 03 — Portfolio construction

Equal-weight 25% portfolio of the top SSR-scored ETF in each of: world · sector · commodity · bond.

Inputs: scores from `notebooks/02_ssr_scoring.ipynb` (persisted to `data/ssr_scores_2016_2026.parquet`).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

import macro_framework as mf

pd.set_option("display.width", 180)

In [2]:
scores_path = Path.cwd().parent / "data" / "ssr_scores_2016_2026.parquet"
scored = pd.read_parquet(scores_path)
print(f"loaded {len(scored)} scored ETFs from {scores_path.name}")

loaded 106 scored ETFs from ssr_scores_2016_2026.parquet


## 1. Top 3 per target category (context)

In [3]:
CATEGORIES = ["world", "sector", "commodity", "bond"]

candidates = (
    scored.loc[scored["category"].isin(CATEGORIES)]
    .groupby("category", group_keys=True)
    .head(3)
    .loc[:, ["symbol", "name", "category", "ssr", "score"]]
)
candidates.sort_values(["category", "ssr"], ascending=[True, False])

,symbol,name,category,ssr,score
83,BIL,SPDR Bloomberg 1-3 Month T-Bill ETF,bond,0.032152,21.698113
84,HYG,iShares iBoxx $ High Yield Corporate Bond ETF,bond,0.030704,20.754717
85,JNK,SPDR Bloomberg High Yield Bond ETF,bond,0.018260,19.811321
39,IAU,iShares Gold Trust,commodity,0.087236,63.207547
55,GDX,VanEck Gold Miners ETF,commodity,0.065618,48.113208
63,GDXJ,VanEck Junior Gold Miners ETF,commodity,0.057200,40.566038
1,XLK,Technology Select Sector SPDR Fund,sector,0.144515,99.056604
4,SMH,VanEck Semiconductor ETF,sector,0.126216,96.226415
11,SOXX,iShares Semiconductor ETF,sector,0.122312,89.622642
0,SWDA.L,iShares Core MSCI World UCITS ETF,world,0.152500,100.000000


## 2. Selected portfolio — top 1 per category, equal-weight

In [4]:
portfolio = mf.select_top_per_category(scored, categories=CATEGORIES)
portfolio[["symbol", "name", "category", "ssr", "score", "weight"]]

,symbol,name,category,ssr,score,weight
0,SWDA.L,iShares Core MSCI World UCITS ETF,world,0.152500,100.000000,0.25
1,XLK,Technology Select Sector SPDR Fund,sector,0.144515,99.056604,0.25
2,IAU,iShares Gold Trust,commodity,0.087236,63.207547,0.25
3,BIL,SPDR Bloomberg 1-3 Month T-Bill ETF,bond,0.032152,21.698113,0.25


In [5]:
print(f"portfolio SSR (weighted avg):  {(portfolio['ssr'] * portfolio['weight']).sum():.4f}")
print(f"portfolio score (weighted avg): {(portfolio['score'] * portfolio['weight']).sum():.1f}")
print(f"weights sum:                    {portfolio['weight'].sum():.4f}")

portfolio SSR (weighted avg):  0.1041
portfolio score (weighted avg): 71.0
weights sum:                    1.0000


## 3. Persist selection

In [6]:
out = Path.cwd().parent / "data" / "portfolio_ssr_top_per_category.parquet"
portfolio.to_parquet(out, index=False)
print(f"wrote {out.relative_to(Path.cwd().parent)}  ({len(portfolio)} positions)")

wrote data/portfolio_ssr_top_per_category.parquet  (4 positions)
